In [4]:
import cv2
import numpy as np
import json
from shapely.geometry import Polygon
import triangle as tr

def process_physics_image(image_path, tolerance=2.0):
    # 1. 이미지 로드 (알파 채널 포함)
    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    if img is None or img.shape[2] < 4:
        return None # 알파 채널이 없는 경우 스킵

    # 2. 알파 채널만 분리하여 마스크 생성
    alpha_channel = img[:, :, 3]
    _, thresh = cv2.threshold(alpha_channel, 1, 255, cv2.THRESH_BINARY)

    # 3. 윤곽선(Contour) 추출
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
        
    # 가장 큰 윤곽선 선택 (지형 본체)
    main_contour = max(contours, key=cv2.contourArea)
    
    # 4. 정점 수 최적화 (Douglas-Peucker 알고리즘)
    # tolerance 값이 클수록 꼭짓점 수가 줄어들어 물리 연산이 가벼워집니다.
    epsilon = tolerance * cv2.arcLength(main_contour, True) / 100.0
    approx_polygon = cv2.approxPolyDP(main_contour, epsilon, True)
    
    # 데이터 포맷 변경 (N, 2)
    points = approx_polygon.reshape(-1, 2).astype(np.float32)
    
    # 5. 오목 다각형을 삼각형(볼록 다각형)들로 분할 (Convex Decomposition)
    # Triangle 라이브러리를 이용해 내부를 삼각망으로 쪼갭니다.
    segments = np.array([[i, (i+1)%len(points)] for i in range(len(points))])
    poly_data = dict(vertices=points, segments=segments)
    triangulated = tr.triangulate(poly_data, 'p') # 'p' 옵션은 다각형 내부만 채움
    
    triangles = []
    for tri in triangulated['triangles']:
        # 삼각형의 세 꼭짓점 좌표 추출
        p1 = triangulated['vertices'][tri[0]].tolist()
        p2 = triangulated['vertices'][tri[1]].tolist()
        p3 = triangulated['vertices'][tri[2]].tolist()
        triangles.append([p1[0], p1[1], p2[0], p2[1], p3[0], p3[1]])
        
    return triangles

In [12]:
# 실행 및 JSON 저장 예시
image_files = ["images/sprD1bApple.png", "images/sprD1bSlicedApple.png"] # 여기에 수백 장의 리스트 바인딩 가능

for img_file in image_files:
    polygons = process_physics_image(img_file, tolerance=0.2)
    if polygons:
        result_data = {}
        file_name = img_file.split("/")[-1]
        result_data[file_name] = polygons
        with open(f"results/{file_name}.json", "w") as f:
            json.dump(result_data, f, indent=4)

